# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [3]:
# TODO: Load environment variables
load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")

### VectorDB Instance

In [4]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="/workspace/Code/project/chromadb")

### Collection

In [7]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it

# embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
#     model_name = "text-embedding-3-large",
#     api_key = openai_key,
#     api_base = "https://openai.vocareum.com/v1"
#     )


client = OpenAI(
    api_key=openai_key,
    base_url="https://openai.vocareum.com/v1"
)


In [9]:
chroma_client.delete_collection(name="udaplay")

In [10]:
collection = chroma_client.create_collection(
    name="udaplay"
)


git clone https://github.com/stantaov/agentic-ai.git

### Add documents

In [11]:
# Make sure you have a directory "project/starter/games"
data_dir = "/workspace/Code/project/starter/games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    doc_id = os.path.splitext(file_name)[0]

    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=[content]
    )

    embedding = response.data[0].embedding

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game],
        embeddings=[embedding]
    )